In [ ]:
# =============================================================================
# CELL 1 — GLOBAL CONFIGURATION
# =============================================================================
from pathlib import Path
import os
import re
import json
import itertools
import traceback
import warnings
import subprocess
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 1) Path to the final Triple notebook
# -----------------------------------------------------------------------------
TRIPLE_NOTEBOOK_PATH = Path("Triple.ipynb")
AUTO_FIND_TRIPLE_NOTEBOOK = False

# -----------------------------------------------------------------------------
# 2) Input base datasets
# -----------------------------------------------------------------------------
# Recommended: explicitly list the 5 base datasets here.
RAW_DATASET_FILES = [
    # EDIT THESE 5 PATHS:
    # "output/single_datasets_report_only/Pre_M1.csv",
    # "output/single_datasets_report_only/Post1_M1.csv",
    # "output/single_datasets_report_only/Post2_M1.csv",
    # "output/single_datasets_report_only/ADC_M1.csv",
    # "output/single_datasets_report_only/T2_M1.csv",
]

# If RAW_DATASET_FILES is empty, auto-discover CSVs from this folder.
AUTO_DISCOVER_DATASETS = True
SOURCE_DATASET_DIR = Path("../Dataset/Breast-Data/Mask1")
DATASET_GLOB_PATTERN = "*.csv"

# Five datasets produce C(5, 3) = 10 triple combinations.
REQUIRE_EXACTLY_FIVE_DATASETS = True

# -----------------------------------------------------------------------------
# 3) Column names and alignment policy
# -----------------------------------------------------------------------------
LABEL_COL = "Label"
GROUP_COL = "PatientID"

ROI_CANDIDATE_COLUMNS = [
    "INFO_NameOfRoi",
    "NameOfRoi",
    "ROI",
    "ROI_Name",
    "LesionID",
    "Lesion_ID",
]

# The first key set that exists and is unique in all three datasets is used.
MERGE_KEY_PRIORITY = [
    ["INFO_NameOfRoi"],
    ["PatientID", "INFO_NameOfRoi"],
    ["LesionID"],
    ["Lesion_ID"],
    ["ROI_Name"],
]

# Row-order fallback is used only when all three row counts and PatientID orders match.
ALLOW_ROW_ORDER_FALLBACK = True
MERGE_HOW = "inner"

# -----------------------------------------------------------------------------
# 4) Output locations
# -----------------------------------------------------------------------------
TRIPLE_DATASET_DIR = Path("output/all_10_triple_concat_datasets")
BATCH_ROOT = Path("nested_cv_outputs_all_triple_concat_batch_smote_compare")
BATCH_RUN_ROOT = Path("batch_all_10_triple_runner_artifacts_smote_compare")

EXECUTED_NOTEBOOK_DIR = BATCH_RUN_ROOT / "executed_triple_notebooks"
MISMATCH_REPORT_DIR = BATCH_RUN_ROOT / "label_group_mismatch_reports"
SUMMARY_DIR = BATCH_RUN_ROOT / "summary_tables"

for p in [TRIPLE_DATASET_DIR, BATCH_ROOT, BATCH_RUN_ROOT, EXECUTED_NOTEBOOK_DIR, MISMATCH_REPORT_DIR, SUMMARY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 5) Notebook execution options
# -----------------------------------------------------------------------------
EXECUTE_TRIPLE_NOTEBOOKS = True
SKIP_EXISTING_SUCCESSFUL_RUNS = False
OVERWRITE_FS_RUNS = True
COMPARE_SMOTE_OPTIONS = True
RUN_SMOTE_ABLATION = False
RUN_SMOTE_VS_BASELINE_COMPARISON = False
KERNEL_NAME = "python3"
NOTEBOOK_TIMEOUT_SECONDS = 43200  # 12 hours per notebook cell
PROCESS_TIMEOUT_SECONDS = 46800   # 13 hours for one complete triple run

ADD_PARAMETERS_TAG_TO_TRIPLE = True
PATCH_FINAL_ZIP_OUTPUT_IN_PLACE = True

print("Configuration loaded.")
print(f"TRIPLE_DATASET_DIR: {TRIPLE_DATASET_DIR.resolve()}")
print(f"BATCH_ROOT:         {BATCH_ROOT.resolve()}")
print(f"SUMMARY_DIR:        {SUMMARY_DIR.resolve()}")

In [ ]:
# =============================================================================
# CELL 2 — ONE-TIME DEPENDENCY CHECK
# =============================================================================

def ensure_import(import_name, pip_name=None):
    import importlib.util
    if importlib.util.find_spec(import_name) is None:
        pip_name = pip_name or import_name
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"OK: {import_name}")

ensure_import("nbformat", "nbformat")
ensure_import("papermill", "papermill")
ensure_import("statsmodels", "statsmodels")
ensure_import("imblearn", "imbalanced-learn")

import nbformat
import papermill as pm

print("Dependency check completed.")

In [ ]:
# =============================================================================
# CELL 3 — FILE DISCOVERY AND VALIDATION
# =============================================================================

def find_triple_notebook(path: Path) -> Path:
    if path.exists():
        return path.resolve()

    if not AUTO_FIND_TRIPLE_NOTEBOOK:
        raise FileNotFoundError(f"Triple notebook not found: {path}")

    candidates = []
    for root in [Path.cwd(), Path("/kaggle/input"), Path("/mnt/data")]:
        if root.exists():
            try:
                candidates.extend(root.rglob(TRIPLE_NOTEBOOK_PATH.name))
            except Exception:
                pass

    candidates = sorted(set(candidates))
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not find {TRIPLE_NOTEBOOK_PATH.name}. Put it next to this notebook or set TRIPLE_NOTEBOOK_PATH."
        )

    print(f"Found {TRIPLE_NOTEBOOK_PATH.name} candidates:")
    for c in candidates:
        print("  -", c)

    return candidates[0].resolve()


def discover_dataset_files():
    if len(RAW_DATASET_FILES) > 0:
        files = [Path(x) for x in RAW_DATASET_FILES]
    else:
        if not AUTO_DISCOVER_DATASETS:
            raise ValueError("RAW_DATASET_FILES is empty and AUTO_DISCOVER_DATASETS=False.")
        if not SOURCE_DATASET_DIR.exists():
            raise FileNotFoundError(
                f"SOURCE_DATASET_DIR does not exist: {SOURCE_DATASET_DIR}\n"
                "Either create this folder or fill RAW_DATASET_FILES manually."
            )
        files = sorted(SOURCE_DATASET_DIR.glob(DATASET_GLOB_PATTERN))

    filtered = []
    excluded_terms = ["pairwise", "triple", "quad", "summary", "comparison", "random_label"]
    for p in files:
        if any(term in p.stem.lower() for term in excluded_terms):
            continue
        filtered.append(p)

    files = [p.resolve() for p in filtered]
    missing = [str(p) for p in files if not p.exists()]
    if missing:
        raise FileNotFoundError("These dataset files were not found:\n" + "\n".join(missing))

    if REQUIRE_EXACTLY_FIVE_DATASETS and len(files) != 5:
        raise ValueError(
            f"Expected exactly 5 base datasets to create 10 triples, but found {len(files)}.\n"
            "Files found:\n" + "\n".join(str(p) for p in files) + "\n\n"
            "Fix RAW_DATASET_FILES or SOURCE_DATASET_DIR."
        )

    if len(files) < 3:
        raise ValueError(f"At least 3 base datasets are required; found {len(files)}.")

    return files


TRIPLE_NOTEBOOK_PATH = find_triple_notebook(TRIPLE_NOTEBOOK_PATH)
DATASET_FILES = discover_dataset_files()

print("Triple notebook:")
print(" ", TRIPLE_NOTEBOOK_PATH)

print("\nBase datasets:")
for i, f in enumerate(DATASET_FILES, start=1):
    print(f"{i}. {f}")

expected_triples = len(list(itertools.combinations(DATASET_FILES, 3)))
print(f"\nNumber of concat triples to create: {expected_triples}")

# -----------------------------------------------------------------------------
# Strict validation of the selected final Triple notebook
# -----------------------------------------------------------------------------
_target_triple_nb = nbformat.read(
    TRIPLE_NOTEBOOK_PATH,
    as_version=4,
)

if len(_target_triple_nb.cells) == 0:
    raise RuntimeError(
        "The selected Triple notebook has no cells."
    )

_parameter_tags = (
    _target_triple_nb.cells[0]
    .metadata
    .get("tags", [])
)

_parameter_source = (
    _target_triple_nb.cells[0].source
)

_required_parameter_names = [
    "INPUT_FILE",
    "DATASET_TAG",
    "BATCH_ROOT",
    "OUTPUT_DIR",
    "TARGET_COLUMN",
    "DATASET_OUTPUT_ROOT",
    "CLEANED_DATASET_PATH",
    "IS_BATCH_RUN",
    "PAPERMILL_BATCH_RUN",
    "BATCH_MODE",
    "RUN_PUBLICATION_PLOTS",
    "RUN_DESCRIPTIVE_PLOTS",
    "RUN_MORPHOLOGICAL_DESCRIPTION",
]

_missing_parameter_names = [
    name
    for name in _required_parameter_names
    if re.search(
        rf"^\s*{re.escape(name)}\s*=",
        _parameter_source,
        flags=re.MULTILINE,
    ) is None
]

_target_code_sources = [
    cell.source
    for cell in _target_triple_nb.cells
    if cell.cell_type == "code"
]

_has_nested_cv = any(
    "def run_nested_cv_experiment" in source
    for source in _target_code_sources
)

_has_patient_cluster_ci = any(
    "def patient_cluster_bootstrap_cis" in source
    for source in _target_code_sources
)

_has_triple_volume_resolution = any(
    "TRIPLE-SAFE VOLUME FEATURE RESOLUTION"
    in source
    for source in _target_code_sources
)

_has_prefix_aware_family_detection = any(
    "def detect_radiomics_family" in source
    for source in _target_code_sources
)

if "parameters" not in _parameter_tags:
    raise RuntimeError(
        "The first cell of the final Triple notebook "
        "must have the Papermill 'parameters' tag."
    )

if _missing_parameter_names:
    raise RuntimeError(
        "Missing required Papermill parameters: "
        + ", ".join(_missing_parameter_names)
    )

if not (
    _has_nested_cv
    and _has_patient_cluster_ci
    and _has_triple_volume_resolution
    and _has_prefix_aware_family_detection
):
    raise RuntimeError(
        "The selected Triple notebook is not the final "
        "batch-safe version.\n"
        f"Selected notebook: {TRIPLE_NOTEBOOK_PATH}"
    )

print(
    "\nTriple batch-safe notebook validation: PASSED"
)

print(
    "Target notebook used by Papermill:"
)

print(
    " ",
    TRIPLE_NOTEBOOK_PATH,
)


In [ ]:
# =============================================================================
# CELL 4 — DATASET READING, TAGGING, AND TRIPLE CONCAT FUNCTIONS
# =============================================================================

def clean_tag_from_path(path):
    """Create a compact dataset tag from filename."""
    stem = Path(path).stem
    suffixes = [
        "_Cleaned", "_cleaned", "_M1_triple", "_triple",
        "_M1_pairwise", "_pairwise", "_M1"
    ]
    changed = True
    while changed:
        changed = False
        for suffix in suffixes:
            if stem.endswith(suffix):
                stem = stem[:-len(suffix)]
                changed = True
    return re.sub(r"[^A-Za-z0-9]+", "_", stem).strip("_")


def read_csv_flexible(path):
    """Read comma-, semicolon-, tab-, or whitespace-separated CSV robustly."""
    attempts = [
        dict(sep=None, engine="python"),
        dict(sep=","),
        dict(sep=";"),
        dict(sep="\t"),
        dict(sep=r"\s+", engine="python"),
    ]
    last_error = None
    for kwargs in attempts:
        try:
            df = pd.read_csv(path, **kwargs)
            if df.shape[1] > 1:
                return df
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Could not read CSV correctly: {path}\nLast error: {last_error}")


def normalize_column_lookup(df):
    return {str(c).strip().lower(): c for c in df.columns}


def resolve_column(df, preferred, candidates=None, required=True, role_name="column"):
    lookup = normalize_column_lookup(df)
    for term in [preferred] + list(candidates or []):
        if term is not None and str(term).strip().lower() in lookup:
            return lookup[str(term).strip().lower()]
    if required:
        raise ValueError(
            f"Could not find required {role_name}. Preferred={preferred}, candidates={candidates}. "
            f"Available columns include: {list(df.columns[:20])}"
        )
    return None


def standardize_label_column(df):
    detected = resolve_column(
        df, LABEL_COL,
        candidates=["label", "Label", "Class", "class", "target", "Target", "diagnosis"],
        required=True, role_name="label column"
    )
    return df.rename(columns={detected: LABEL_COL}) if detected != LABEL_COL else df.copy()


def find_first_existing_column(df, candidates):
    lookup = normalize_column_lookup(df)
    for candidate in candidates:
        key = str(candidate).strip().lower()
        if key in lookup:
            return lookup[key]
    return None


def derive_patient_id(df):
    df = df.copy()
    if GROUP_COL in df.columns and df[GROUP_COL].notna().any():
        source = df[GROUP_COL].astype(str)
    else:
        roi_col = find_first_existing_column(df, ROI_CANDIDATE_COLUMNS)
        patient_col = find_first_existing_column(
            df, ["INFO_PatientName", "PatientName", "patient_name", "SubjectID", "CaseID", "ID"]
        )
        if roi_col is not None:
            source = df[roi_col].astype(str)
        elif patient_col is not None:
            source = df[patient_col].astype(str)
        else:
            raise ValueError(
                "Could not derive PatientID. Provide PatientID, INFO_NameOfRoi, or INFO_PatientName."
            )

    extracted = source.str.extract(r"(P\d+)", expand=False)
    fallback = source.str.replace(r"^[A-Za-z0-9]+_D_", "", regex=True)
    fallback = fallback.str.replace(r"[^A-Za-z0-9]+", "_", regex=True).str.strip("_")
    df[GROUP_COL] = extracted.fillna(fallback).astype(str)
    return df


def preprocess_base_dataset(path):
    df = read_csv_flexible(path).copy()
    empty_cols = df.columns[df.isna().all()].tolist()
    if empty_cols:
        df = df.drop(columns=empty_cols)
    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed:")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    df = standardize_label_column(df)
    df = derive_patient_id(df)
    return df


def choose_merge_keys_many(datasets):
    """Choose one safe merge key that is present and unique in all three datasets."""
    candidate_sets = list(MERGE_KEY_PRIORITY)
    for roi_col in ROI_CANDIDATE_COLUMNS:
        candidate_sets.append([roi_col])
        candidate_sets.append([GROUP_COL, roi_col])
    candidate_sets.append([GROUP_COL])

    seen = set()
    for keys in candidate_sets:
        keys = [k for k in keys if k is not None]
        key_tuple = tuple(keys)
        if key_tuple in seen:
            continue
        seen.add(key_tuple)
        if all(all(k in df.columns for k in keys) for df in datasets):
            if all(int(df.duplicated(subset=keys).sum()) == 0 for df in datasets):
                return keys, "unique_key"

    if ALLOW_ROW_ORDER_FALLBACK:
        same_length = len({len(df) for df in datasets}) == 1
        if same_length:
            base_order = datasets[0][GROUP_COL].astype(str).reset_index(drop=True)
            same_order = all(
                base_order.equals(df[GROUP_COL].astype(str).reset_index(drop=True))
                for df in datasets[1:]
            )
            if same_order:
                for df in datasets:
                    df["__row_index_triple_key__"] = np.arange(len(df))
                return ["__row_index_triple_key__"], "row_order_fallback"

    raise ValueError(
        "No safe common unique merge key was found across all three datasets. "
        "Use a unique ROI/Lesion column or verify identical PatientID row order."
    )


def prepare_dataset_for_triple_merge(df, tag, keys):
    """Keep merge keys shared; tag all other metadata and features."""
    df = df.copy()
    metadata_candidates = [GROUP_COL, LABEL_COL, "INFO_PatientName"] + ROI_CANDIDATE_COLUMNS
    meta_cols = list(keys)
    for col in metadata_candidates:
        if col in df.columns and col not in meta_cols:
            meta_cols.append(col)

    feature_cols = [c for c in df.columns if c not in meta_cols]
    rename_map = {}
    for col in meta_cols:
        if col not in keys:
            rename_map[col] = f"{col}__{tag}"
    for col in feature_cols:
        rename_map[col] = f"{tag}__{col}"

    prepared = df[meta_cols + feature_cols].rename(columns=rename_map)
    tagged_features = [rename_map[c] for c in feature_cols]
    return prepared, tagged_features


def tagged_metadata_column(base_col, tag, keys, merged):
    if base_col in keys and base_col in merged.columns:
        return base_col
    candidate = f"{base_col}__{tag}"
    return candidate if candidate in merged.columns else None


def mismatch_mask_against_first(merged, columns):
    reference = merged[columns[0]].astype(str).str.strip()
    mask = pd.Series(False, index=merged.index)
    for col in columns[1:]:
        mask |= reference.ne(merged[col].astype(str).str.strip())
    return mask


def build_concat_triple_dataset(path_a, path_b, path_c):
    paths = [Path(path_a), Path(path_b), Path(path_c)]
    tags = [clean_tag_from_path(path) for path in paths]
    triple_tag = "_".join(tags)
    output_path = TRIPLE_DATASET_DIR / f"{triple_tag}_M1_triple.csv"

    datasets = [preprocess_base_dataset(path) for path in paths]
    keyed_datasets = [df.copy() for df in datasets]
    keys, key_mode = choose_merge_keys_many(keyed_datasets)

    prepared = []
    feature_lists = []
    for df, tag in zip(keyed_datasets, tags):
        pre, features = prepare_dataset_for_triple_merge(df, tag, keys)
        prepared.append(pre)
        feature_lists.append(features)

    merged = prepared[0]
    for next_df in prepared[1:]:
        merged = merged.merge(next_df, on=keys, how=MERGE_HOW, validate="one_to_one")

    if len(merged) == 0:
        raise ValueError(
            f"{triple_tag}: merge produced 0 rows. Check keys={keys} and MERGE_HOW={MERGE_HOW}."
        )

    label_cols = [tagged_metadata_column(LABEL_COL, tag, keys, merged) for tag in tags]
    if any(col is None for col in label_cols):
        raise ValueError(f"{triple_tag}: could not identify all three label columns after merge.")

    label_mismatch = mismatch_mask_against_first(merged, label_cols)
    if label_mismatch.any():
        report_path = MISMATCH_REPORT_DIR / f"{triple_tag}_label_mismatches.csv"
        merged.loc[label_mismatch, keys + label_cols].to_csv(report_path, index=False)
        raise ValueError(
            f"Label mismatch detected for triple {triple_tag}: {int(label_mismatch.sum())} rows. "
            f"Report saved to: {report_path}"
        )

    group_cols = [tagged_metadata_column(GROUP_COL, tag, keys, merged) for tag in tags]
    existing_group_cols = [col for col in group_cols if col is not None]
    if len(existing_group_cols) >= 2:
        group_mismatch = mismatch_mask_against_first(merged, existing_group_cols)
        if group_mismatch.any():
            report_path = MISMATCH_REPORT_DIR / f"{triple_tag}_group_mismatches.csv"
            merged.loc[group_mismatch, keys + existing_group_cols].to_csv(report_path, index=False)
            raise ValueError(
                f"PatientID mismatch detected for triple {triple_tag}: {int(group_mismatch.sum())} rows. "
                f"Report saved to: {report_path}"
            )

    final_df = pd.DataFrame(index=merged.index)
    if existing_group_cols:
        final_df[GROUP_COL] = merged[existing_group_cols[0]].astype(str)
    else:
        final_df[GROUP_COL] = merged[keys[0]].astype(str)

    # Preserve one representative ROI and patient-name metadata column.
    roi_source = None
    for roi_candidate in ROI_CANDIDATE_COLUMNS:
        if roi_candidate in keys and roi_candidate in merged.columns:
            roi_source = roi_candidate
            break
        candidate = tagged_metadata_column(roi_candidate, tags[0], keys, merged)
        if candidate is not None:
            roi_source = candidate
            break
    if roi_source is not None:
        final_df["INFO_NameOfRoi"] = merged[roi_source].astype(str)

    patient_name_source = tagged_metadata_column("INFO_PatientName", tags[0], keys, merged)
    if patient_name_source is not None:
        final_df["INFO_PatientName"] = merged[patient_name_source].astype(str)

    for feature_list in feature_lists:
        for col in feature_list:
            if col in merged.columns:
                final_df[col] = merged[col]

    final_df[LABEL_COL] = pd.to_numeric(merged[label_cols[0]], errors="raise").astype(int)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(output_path, index=False)

    label_counts = final_df[LABEL_COL].value_counts().sort_index().to_dict()
    numeric_feature_count = len([
        c for c in final_df.select_dtypes(include=[np.number]).columns if c != LABEL_COL
    ])

    meta = {
        "Triple_Tag": triple_tag,
        "Dataset_A": tags[0],
        "Dataset_B": tags[1],
        "Dataset_C": tags[2],
        "Dataset_A_Path": str(paths[0]),
        "Dataset_B_Path": str(paths[1]),
        "Dataset_C_Path": str(paths[2]),
        "Triple_Input_File": str(output_path),
        "Merge_Keys": ",".join(keys),
        "Merge_Key_Mode": key_mode,
        "Merge_How": MERGE_HOW,
        "Rows_A": int(len(datasets[0])),
        "Rows_B": int(len(datasets[1])),
        "Rows_C": int(len(datasets[2])),
        "Rows_After_Merge": int(len(final_df)),
        "N_Features_A": int(len(feature_lists[0])),
        "N_Features_B": int(len(feature_lists[1])),
        "N_Features_C": int(len(feature_lists[2])),
        "N_Features_Total": int(numeric_feature_count),
        "Label_Counts": json.dumps({str(k): int(v) for k, v in label_counts.items()}, ensure_ascii=False),
        "Status": "dataset_created",
        "Error": "",
    }
    return output_path, meta

In [ ]:
# =============================================================================
# CELL 5 — CREATE ALL 10 CONCAT TRIPLE DATASETS
# =============================================================================

triple_dataset_rows = []
triple_dataset_paths = []

for path_a, path_b, path_c in itertools.combinations(DATASET_FILES, 3):
    tags = [clean_tag_from_path(path) for path in [path_a, path_b, path_c]]
    triple_tag = "_".join(tags)
    try:
        out_path, meta = build_concat_triple_dataset(path_a, path_b, path_c)
        triple_dataset_paths.append(out_path)
        triple_dataset_rows.append(meta)
        print(
            f"✅ Created: {meta['Triple_Tag']} | rows={meta['Rows_After_Merge']} | "
            f"features={meta['N_Features_Total']} | keys={meta['Merge_Keys']}"
        )
    except Exception as exc:
        triple_dataset_rows.append({
            "Triple_Tag": triple_tag,
            "Dataset_A": tags[0],
            "Dataset_B": tags[1],
            "Dataset_C": tags[2],
            "Dataset_A_Path": str(path_a),
            "Dataset_B_Path": str(path_b),
            "Dataset_C_Path": str(path_c),
            "Triple_Input_File": "",
            "Merge_Keys": "",
            "Merge_Key_Mode": "",
            "Merge_How": MERGE_HOW,
            "Rows_A": np.nan,
            "Rows_B": np.nan,
            "Rows_C": np.nan,
            "Rows_After_Merge": np.nan,
            "N_Features_A": np.nan,
            "N_Features_B": np.nan,
            "N_Features_C": np.nan,
            "N_Features_Total": np.nan,
            "Label_Counts": "",
            "Status": "dataset_creation_failed",
            "Error": str(exc),
        })
        print(f"❌ Failed to create triple dataset: {triple_tag}")
        print(str(exc))


df_triple_datasets = pd.DataFrame(triple_dataset_rows)
dataset_log_path = SUMMARY_DIR / "triple_concat_dataset_creation_log.csv"
df_triple_datasets.to_csv(dataset_log_path, index=False)

print("\nSaved dataset creation log:")
print(dataset_log_path)
display(df_triple_datasets)

In [ ]:
# =============================================================================
# CELL 6 — RUN THE FINAL TRIPLE NOTEBOOK FOR ALL 10 TRIPLES
# =============================================================================
# Every triple is executed in a separate Python subprocess.
# Papermill progress and notebook output are displayed live and saved to logs.


def validate_triple_notebook_for_execution(
    notebook_path,
):
    notebook_path = Path(
        notebook_path
    ).resolve()

    if not notebook_path.exists():
        raise FileNotFoundError(
            "Triple notebook not found:\n"
            f"{notebook_path}"
        )

    notebook = nbformat.read(
        notebook_path,
        as_version=4,
    )

    if (
        len(notebook.cells) == 0
        or "parameters"
        not in notebook.cells[0]
        .metadata
        .get("tags", [])
    ):
        raise ValueError(
            "Triple notebook must have a first "
            "cell tagged 'parameters'."
        )

    return notebook_path


def final_triple_summary_exists(
    triple_tag,
):
    summary_path = (
        BATCH_ROOT
        / triple_tag
        / "fs_method_comparison"
        / "_comparison_summary"
        / "publication_fs_comparison_best_per_method.csv"
    )

    return summary_path.exists()


def build_triple_parameters(
    triple_input_file,
    triple_tag,
):
    triple_input_file = Path(
        triple_input_file
    ).resolve()

    output_dir = (
        Path(
            "output/"
            "cleaned_triple_all_10_smote_compare"
        )
        / triple_tag
    )

    dataset_output_root = (
        BATCH_ROOT
        / triple_tag
    )

    cleaned_dataset_path = (
        output_dir
        / (
            f"{triple_input_file.stem}"
            "_Cleaned.csv"
        )
    )

    return {
        "INPUT_FILE": str(
            triple_input_file
        ),

        "DATASET_TAG": str(
            triple_tag
        ),

        "BATCH_ROOT": str(
            BATCH_ROOT
        ),

        "OUTPUT_DIR": str(
            output_dir
        ),

        "TARGET_COLUMN": str(
            LABEL_COL
        ),

        "DATASET_OUTPUT_ROOT": str(
            dataset_output_root
        ),

        "CLEANED_DATASET_PATH": str(
            cleaned_dataset_path
        ),

        "OVERWRITE_FS_RUNS": bool(
            OVERWRITE_FS_RUNS
        ),

        "COMPARE_SMOTE_OPTIONS": bool(
            COMPARE_SMOTE_OPTIONS
        ),

        "TARGET_ABLATION": None,

        "RUN_SMOTE_ABLATION": bool(
            RUN_SMOTE_ABLATION
        ),

        "RUN_SMOTE_VS_BASELINE_COMPARISON": bool(
            RUN_SMOTE_VS_BASELINE_COMPARISON
        ),

        # Batch execution flags
        "IS_BATCH_RUN": True,
        "PAPERMILL_BATCH_RUN": True,
        "BATCH_MODE": True,

        # Report-only cells are skipped
        "RUN_PUBLICATION_PLOTS": False,
        "RUN_DESCRIPTIVE_PLOTS": False,
        "RUN_MORPHOLOGICAL_DESCRIPTION": False,
        "MASK1_SOURCE_DIR": None,
    }


TRIPLE_EXEC_NOTEBOOK_PATH = (
    validate_triple_notebook_for_execution(
        TRIPLE_NOTEBOOK_PATH
    )
)


PAPERMILL_CHILD_CODE = r"""
import json
import sys

import papermill as pm

input_path = sys.argv[1]
output_path = sys.argv[2]
parameters_path = sys.argv[3]
kernel_name = sys.argv[4]
cwd = sys.argv[5]
execution_timeout_text = sys.argv[6]

with open(
    parameters_path,
    "r",
    encoding="utf-8",
) as handle:
    parameters = json.load(handle)

execution_timeout = (
    None
    if execution_timeout_text == "None"
    else int(execution_timeout_text)
)

pm.execute_notebook(
    input_path=input_path,
    output_path=output_path,
    parameters=parameters,
    kernel_name=kernel_name,
    cwd=cwd,
    progress_bar=True,
    log_output=True,
    request_save_on_cell_execute=True,
    execution_timeout=execution_timeout,
)
"""


def terminate_child_process(
    process,
    grace_seconds=10,
):
    if (
        process is None
        or process.poll() is not None
    ):
        return

    process.terminate()

    try:
        process.wait(
            timeout=grace_seconds
        )

    except subprocess.TimeoutExpired:
        process.kill()

        process.wait(
            timeout=grace_seconds
        )


run_rows = []


if EXECUTE_TRIPLE_NOTEBOOKS:

    for _, row in (
        df_triple_datasets.iterrows()
    ):

        triple_tag = row["Triple_Tag"]


        if row["Status"] != "dataset_created":

            run_rows.append({
                "Triple_Tag": triple_tag,
                "Run_Status":
                    "not_run_dataset_creation_failed",
                "Executed_Notebook": "",
                "Parameters_File": "",
                "Stdout_Log": "",
                "Stderr_Log": "",
                "Return_Code": np.nan,
                "Error": row.get(
                    "Error",
                    "",
                ),
                "Started_At": "",
                "Finished_At": "",
            })

            continue


        triple_input_file = (
            row["Triple_Input_File"]
        )


        if (
            SKIP_EXISTING_SUCCESSFUL_RUNS
            and final_triple_summary_exists(
                triple_tag
            )
        ):

            print(
                "Skipping existing successful run: "
                f"{triple_tag}"
            )

            run_rows.append({
                "Triple_Tag": triple_tag,
                "Run_Status": "skipped_existing",
                "Executed_Notebook": "",
                "Parameters_File": "",
                "Stdout_Log": "",
                "Stderr_Log": "",
                "Return_Code": 0,
                "Error": "",
                "Started_At": "",
                "Finished_At": "",
            })

            continue


        parameters = build_triple_parameters(
            triple_input_file,
            triple_tag,
        )


        executed_path = (
            EXECUTED_NOTEBOOK_DIR
            / (
                f"{triple_tag}"
                "__Triple_executed.ipynb"
            )
        )

        parameters_path = (
            EXECUTED_NOTEBOOK_DIR
            / (
                f"{triple_tag}"
                "__papermill_parameters.json"
            )
        )

        stdout_path = (
            EXECUTED_NOTEBOOK_DIR
            / (
                f"{triple_tag}"
                "__papermill_stdout.log"
            )
        )

        stderr_path = (
            EXECUTED_NOTEBOOK_DIR
            / (
                f"{triple_tag}"
                "__papermill_stderr.log"
            )
        )


        parameters_path.write_text(
            json.dumps(
                parameters,
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )


        command = [
            sys.executable,
            "-c",
            PAPERMILL_CHILD_CODE,
            str(
                TRIPLE_EXEC_NOTEBOOK_PATH
            ),
            str(
                executed_path
            ),
            str(
                parameters_path
            ),
            str(
                KERNEL_NAME
            ),
            str(
                Path.cwd()
            ),
            (
                "None"
                if NOTEBOOK_TIMEOUT_SECONDS
                is None
                else str(
                    NOTEBOOK_TIMEOUT_SECONDS
                )
            ),
        ]


        started = (
            datetime.now()
            .isoformat(
                timespec="seconds"
            )
        )


        print(
            "\n"
            + "=" * 100
        )

        print(
            f"RUNNING TRIPLE: {triple_tag}"
        )

        print(
            "Input:",
            triple_input_file,
        )

        print(
            "Output root:",
            parameters[
                "DATASET_OUTPUT_ROOT"
            ],
        )

        print(
            "Executed notebook:",
            executed_path,
        )

        print(
            "Live/output log:",
            stdout_path,
        )

        print(
            "=" * 100
        )


        process = None


        try:

            with stdout_path.open(
                "w",
                encoding="utf-8",
                buffering=1,
            ) as live_log_handle:

                process = subprocess.Popen(
                    command,
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    cwd=str(
                        Path.cwd()
                    ),
                    text=True,
                    bufsize=1,
                    universal_newlines=True,
                )

                process_start_time = (
                    time.monotonic()
                )


                while True:

                    if process.stdout is None:
                        break

                    line = (
                        process.stdout.readline()
                    )


                    if line:

                        live_log_handle.write(
                            line
                        )

                        live_log_handle.flush()

                        print(
                            f"[{triple_tag}] "
                            f"{line}",
                            end="",
                            flush=True,
                        )


                    elif process.poll() is not None:
                        break


                    if (
                        PROCESS_TIMEOUT_SECONDS
                        is not None
                        and (
                            time.monotonic()
                            - process_start_time
                        )
                        > PROCESS_TIMEOUT_SECONDS
                    ):
                        raise subprocess.TimeoutExpired(
                            cmd=command,
                            timeout=(
                                PROCESS_TIMEOUT_SECONDS
                            ),
                        )


                return_code = (
                    process.wait()
                )


            stderr_path.write_text(
                (
                    "Child stderr was merged into "
                    "the live stdout log:\n"
                    f"{stdout_path}\n"
                ),
                encoding="utf-8",
            )


            finished = (
                datetime.now()
                .isoformat(
                    timespec="seconds"
                )
            )


            if return_code == 0:

                run_status = "completed"
                error_text = ""

                print(
                    f"Completed: {triple_tag}"
                )

            else:

                run_status = "failed"

                error_text = (
                    "Papermill child process exited "
                    f"with return code {return_code}. "
                    f"See log: {stdout_path}"
                )

                print(
                    f"Failed: {triple_tag}"
                )

                print(
                    error_text
                )


            run_rows.append({
                "Triple_Tag": triple_tag,
                "Run_Status": run_status,
                "Executed_Notebook": (
                    str(
                        executed_path
                    )
                    if executed_path.exists()
                    else ""
                ),
                "Parameters_File": str(
                    parameters_path
                ),
                "Stdout_Log": str(
                    stdout_path
                ),
                "Stderr_Log": str(
                    stderr_path
                ),
                "Return_Code": int(
                    return_code
                ),
                "Error": error_text,
                "Started_At": started,
                "Finished_At": finished,
            })


        except subprocess.TimeoutExpired:

            terminate_child_process(
                process
            )

            finished = (
                datetime.now()
                .isoformat(
                    timespec="seconds"
                )
            )

            error_text = (
                "Triple run exceeded "
                "PROCESS_TIMEOUT_SECONDS="
                f"{PROCESS_TIMEOUT_SECONDS}."
            )

            run_rows.append({
                "Triple_Tag": triple_tag,
                "Run_Status": "timeout",
                "Executed_Notebook": (
                    str(
                        executed_path
                    )
                    if executed_path.exists()
                    else ""
                ),
                "Parameters_File": str(
                    parameters_path
                ),
                "Stdout_Log": str(
                    stdout_path
                ),
                "Stderr_Log": str(
                    stderr_path
                ),
                "Return_Code": np.nan,
                "Error": error_text,
                "Started_At": started,
                "Finished_At": finished,
            })

            print(
                f"Timeout: {triple_tag}"
            )

            print(
                error_text
            )


        except KeyboardInterrupt:

            terminate_child_process(
                process
            )

            finished = (
                datetime.now()
                .isoformat(
                    timespec="seconds"
                )
            )

            run_rows.append({
                "Triple_Tag": triple_tag,
                "Run_Status": "interrupted",
                "Executed_Notebook": (
                    str(
                        executed_path
                    )
                    if executed_path.exists()
                    else ""
                ),
                "Parameters_File": str(
                    parameters_path
                ),
                "Stdout_Log": str(
                    stdout_path
                ),
                "Stderr_Log": str(
                    stderr_path
                ),
                "Return_Code": np.nan,
                "Error": "Interrupted by user.",
                "Started_At": started,
                "Finished_At": finished,
            })

            print(
                f"Interrupted: {triple_tag}"
            )

            raise


        except BaseException as error:

            terminate_child_process(
                process
            )

            finished = (
                datetime.now()
                .isoformat(
                    timespec="seconds"
                )
            )

            error_text = (
                traceback.format_exc()
            )

            failure_path = (
                EXECUTED_NOTEBOOK_DIR
                / (
                    f"{triple_tag}"
                    "__FAILED_error.txt"
                )
            )

            failure_path.write_text(
                error_text,
                encoding="utf-8",
            )

            run_rows.append({
                "Triple_Tag": triple_tag,
                "Run_Status":
                    "failed_runner_exception",
                "Executed_Notebook": (
                    str(
                        executed_path
                    )
                    if executed_path.exists()
                    else ""
                ),
                "Parameters_File": str(
                    parameters_path
                ),
                "Stdout_Log": str(
                    stdout_path
                ),
                "Stderr_Log": str(
                    stderr_path
                ),
                "Return_Code": np.nan,
                "Error": str(
                    error
                ),
                "Started_At": started,
                "Finished_At": finished,
            })

            print(
                f"Runner exception: {triple_tag}"
            )

            print(
                str(
                    error
                )
            )


else:

    print(
        "EXECUTE_TRIPLE_NOTEBOOKS=False. "
        "Triple datasets were created but "
        "notebook execution was skipped."
    )


    for _, row in (
        df_triple_datasets.iterrows()
    ):

        run_rows.append({
            "Triple_Tag": row[
                "Triple_Tag"
            ],
            "Run_Status": "not_executed",
            "Executed_Notebook": "",
            "Parameters_File": "",
            "Stdout_Log": "",
            "Stderr_Log": "",
            "Return_Code": np.nan,
            "Error": "",
            "Started_At": "",
            "Finished_At": "",
        })


df_run_log = pd.DataFrame(
    run_rows
)


run_log_path = (
    SUMMARY_DIR
    / "all_10_triple_notebook_run_log.csv"
)


df_run_log.to_csv(
    run_log_path,
    index=False,
)


print(
    "\nRun log saved:"
)

print(
    run_log_path
)


display(
    df_run_log
)


In [ ]:
# =============================================================================
# CELL 7 — COLLECT BEST RESULTS FROM EACH TRIPLE RUN
# =============================================================================

def read_best_result_for_triple(triple_tag):
    triple_root = BATCH_ROOT / triple_tag
    summary_root = triple_root / "fs_method_comparison" / "_comparison_summary"

    ranking_path = summary_root / "fs_method_composite_ranking.csv"
    publication_path = summary_root / "publication_fs_comparison_best_per_method.csv"
    best_per_method_path = summary_root / "fs_comparison_best_model_per_method.csv"
    all_models_path = summary_root / "fs_comparison_all_models.csv"

    if ranking_path.exists():
        df_source = pd.read_csv(ranking_path)
        source_path = ranking_path
        if len(df_source) == 0:
            raise RuntimeError(f"Empty ranking table: {ranking_path}")
        best = df_source.iloc[0].copy()

    elif publication_path.exists():
        df_source = pd.read_csv(publication_path)
        source_path = publication_path
        if len(df_source) == 0:
            raise RuntimeError(f"Empty publication table: {publication_path}")
        sort_cols = [c for c in ["Mean_AUC", "Pooled_AUC", "Mean_PR_AUC"] if c in df_source.columns]
        best = df_source.sort_values(sort_cols, ascending=False).iloc[0].copy() if sort_cols else df_source.iloc[0].copy()

    elif best_per_method_path.exists():
        df_source = pd.read_csv(best_per_method_path)
        source_path = best_per_method_path
        if len(df_source) == 0:
            raise RuntimeError(f"Empty best-per-method table: {best_per_method_path}")
        sort_cols = [c for c in ["Mean_AUC", "Pooled_AUC", "Mean_PR_AUC"] if c in df_source.columns]
        best = df_source.sort_values(sort_cols, ascending=False).iloc[0].copy() if sort_cols else df_source.iloc[0].copy()

    else:
        raise FileNotFoundError(
            f"No comparison summary found for {triple_tag}. Expected one of:\n"
            f"  {ranking_path}\n"
            f"  {publication_path}\n"
            f"  {best_per_method_path}"
        )

    result = {"Triple_Tag": triple_tag, "Result_Source": str(source_path)}

    preferred_cols = [
        "Experiment_ID",
        "FS_METHOD",
        "FS_N_FEATURES",
        "USE_SMOTE",
        "Model",
        "Mean_AUC",
        "Std_AUC",
        "Pooled_AUC",
        "Mean_PR_AUC",
        "Pooled_PR_AUC",
        "Mean_Bal_Acc",
        "Pooled_Bal_Acc",
        "Mean_F1",
        "Pooled_F1",
        "Mean_Specificity",
        "Pooled_Specificity",
        "Avg_Features",
        "N_Unique_Selected_Features",
        "Max_Selection_Count",
        "Mean_Selection_Count",
        "Composite_Rank_Score",
        "Pooled_AUC_Macro_OvR",
        "Pooled_AUC_Micro_OvR",
        "Pooled_AUC_CI95_Low",
        "Pooled_AUC_CI95_High",
        "Std_PR_AUC",
        "Pooled_PR_AUC_CI95_Low",
        "Pooled_PR_AUC_CI95_High",
        "Mean_Acc",
        "Std_Acc",
        "Pooled_Acc",
        "Pooled_Acc_CI95_Low",
        "Pooled_Acc_CI95_High",
        "Std_Bal_Acc",
        "Pooled_Bal_Acc_CI95_Low",
        "Pooled_Bal_Acc_CI95_High",
        "Mean_Prec",
        "Std_Prec",
        "Pooled_Prec",
        "Pooled_Prec_CI95_Low",
        "Pooled_Prec_CI95_High",
        "Mean_Rec",
        "Std_Rec",
        "Pooled_Rec",
        "Pooled_Rec_CI95_Low",
        "Pooled_Rec_CI95_High",
        "Std_F1",
        "Pooled_F1_CI95_Low",
        "Pooled_F1_CI95_High",
        "Std_Specificity",
        "Pooled_Specificity_CI95_Low",
        "Pooled_Specificity_CI95_High",
        "Mean_Kappa",
        "Std_Kappa",
        "Pooled_Kappa",
        "Pooled_Kappa_CI95_Low",
        "Pooled_Kappa_CI95_High",
        "Pooled_Brier",
        "Pooled_Brier_CI95_Low",
        "Pooled_Brier_CI95_High",
        "CI_Bootstrap_Requested",
        "CI_Bootstrap_Min_Successful",
        "Run_Name",
        "Run_Dir",
    ]

    for c in preferred_cols:
        result[c] = best[c] if c in best.index else np.nan

    return result


summary_rows = []
full_best_tables = []
full_all_model_tables = []

for _, dataset_row in df_triple_datasets.iterrows():
    triple_tag = dataset_row["Triple_Tag"]
    base_info = dataset_row.to_dict()

    if dataset_row["Status"] != "dataset_created":
        base_info.update({
            "Final_Status": "not_run_dataset_creation_failed",
            "Run_Status": "not_run_dataset_creation_failed",
            "Result_Error": dataset_row.get("Error", ""),
        })
        summary_rows.append(base_info)
        continue

    run_status = "unknown"
    if "df_run_log" in globals() and len(df_run_log) > 0:
        match = df_run_log[df_run_log["Triple_Tag"] == triple_tag]
        if len(match) > 0:
            run_status = match.iloc[0]["Run_Status"]

    try:
        best_result = read_best_result_for_triple(triple_tag)
        base_info.update(best_result)
        base_info.update({
            "Run_Status": run_status,
            "Final_Status": "success",
            "Result_Error": "",
        })
        summary_rows.append(base_info)

        best_per_method_path = (
            BATCH_ROOT / triple_tag / "fs_method_comparison" / "_comparison_summary" /
            "fs_comparison_best_model_per_method.csv"
        )
        if best_per_method_path.exists():
            df_tmp = pd.read_csv(best_per_method_path)
            df_tmp.insert(0, "Triple_Tag", triple_tag)
            df_tmp.insert(1, "Dataset_A", dataset_row["Dataset_A"])
            df_tmp.insert(2, "Dataset_B", dataset_row["Dataset_B"])
            df_tmp.insert(3, "Dataset_C", dataset_row["Dataset_C"])
            full_best_tables.append(df_tmp)

        all_models_path = (
            BATCH_ROOT / triple_tag / "fs_method_comparison" / "_comparison_summary" /
            "fs_comparison_all_models.csv"
        )
        if all_models_path.exists():
            df_tmp = pd.read_csv(all_models_path)
            df_tmp.insert(0, "Triple_Tag", triple_tag)
            df_tmp.insert(1, "Dataset_A", dataset_row["Dataset_A"])
            df_tmp.insert(2, "Dataset_B", dataset_row["Dataset_B"])
            df_tmp.insert(3, "Dataset_C", dataset_row["Dataset_C"])
            full_all_model_tables.append(df_tmp)

    except Exception as e:
        base_info.update({
            "Run_Status": run_status,
            "Final_Status": "result_collection_failed",
            "Result_Error": str(e),
        })
        summary_rows.append(base_info)


df_all_10_summary = pd.DataFrame(summary_rows)

if "Mean_AUC" in df_all_10_summary.columns:
    df_all_10_summary["_sort_auc"] = pd.to_numeric(df_all_10_summary["Mean_AUC"], errors="coerce")
    df_all_10_summary = df_all_10_summary.sort_values(
        ["Final_Status", "_sort_auc"],
        ascending=[False, False],
        na_position="last",
    ).drop(columns=["_sort_auc"]).reset_index(drop=True)

summary_path = SUMMARY_DIR / "all_10_triple_concat_summary.csv"
df_all_10_summary.to_csv(summary_path, index=False)

print("Main summary saved:")
print(summary_path)

if full_best_tables:
    df_full_best = pd.concat(full_best_tables, ignore_index=True)
    full_best_path = SUMMARY_DIR / "all_10_triple_full_best_per_fs_method.csv"
    df_full_best.to_csv(full_best_path, index=False)
    print("Full best-per-FS-method table saved:")
    print(full_best_path)
else:
    df_full_best = pd.DataFrame()

if full_all_model_tables:
    df_full_all_models = pd.concat(full_all_model_tables, ignore_index=True)
    full_all_models_path = SUMMARY_DIR / "all_10_triple_full_all_models.csv"
    df_full_all_models.to_csv(full_all_models_path, index=False)
    print("Full all-model table saved:")
    print(full_all_models_path)
else:
    df_full_all_models = pd.DataFrame()



# -----------------------------------------------------------------------------
# SMOTE vs no-SMOTE comparison across all 10 triple datasets
# -----------------------------------------------------------------------------
def _to_bool_smote_batch(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return np.nan
    return str(x).strip().lower() in ["true", "1", "yes", "smote"]

def build_batch_smote_delta_table(df, key_cols):
    if df is None or len(df) == 0 or "USE_SMOTE" not in df.columns:
        return pd.DataFrame()

    work = df.copy()
    work["USE_SMOTE_BOOL"] = work["USE_SMOTE"].map(_to_bool_smote_batch)

    metric_cols = [
        "Mean_AUC", "Std_AUC", "Pooled_AUC",
        "Mean_PR_AUC", "Std_PR_AUC", "Pooled_PR_AUC",
        "Mean_Acc", "Std_Acc", "Pooled_Acc",
        "Mean_Bal_Acc", "Std_Bal_Acc", "Pooled_Bal_Acc",
        "Mean_Prec", "Std_Prec", "Pooled_Prec",
        "Mean_Rec", "Std_Rec", "Pooled_Rec",
        "Mean_F1", "Std_F1", "Pooled_F1",
        "Mean_Specificity", "Std_Specificity", "Pooled_Specificity",
        "Mean_Kappa", "Std_Kappa", "Pooled_Kappa",
        "Pooled_Brier",
        "Avg_Features", "N_Unique_Selected_Features"
    ]
    metric_cols = [c for c in metric_cols if c in work.columns]

    rows = []
    for keys, g in work.groupby(key_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        g_no = g[g["USE_SMOTE_BOOL"] == False]
        g_sm = g[g["USE_SMOTE_BOOL"] == True]

        if len(g_no) == 0 or len(g_sm) == 0:
            continue

        no_row = g_no.iloc[0]
        sm_row = g_sm.iloc[0]

        out = {k: v for k, v in zip(key_cols, keys)}

        if "Model" not in key_cols:
            out["Best_Model_No_SMOTE"] = no_row.get("Model", np.nan)
            out["Best_Model_SMOTE"] = sm_row.get("Model", np.nan)

        out["Run_Name_No_SMOTE"] = no_row.get("Run_Name", np.nan)
        out["Run_Name_SMOTE"] = sm_row.get("Run_Name", np.nan)

        for m in metric_cols:
            no_val = pd.to_numeric(no_row.get(m, np.nan), errors="coerce")
            sm_val = pd.to_numeric(sm_row.get(m, np.nan), errors="coerce")
            out[f"{m}_No_SMOTE"] = no_val
            out[f"{m}_SMOTE"] = sm_val
            out[f"Delta_{m}_SMOTE_minus_No_SMOTE"] = sm_val - no_val

        rows.append(out)

    return pd.DataFrame(rows)


if "df_full_best" in globals() and len(df_full_best) > 0:
    df_triple_smote_vs_no_smote_best = build_batch_smote_delta_table(
        df_full_best,
        key_cols=["Triple_Tag", "Dataset_A", "Dataset_B", "Dataset_C", "Experiment_ID", "FS_METHOD"]
    )

    smote_best_path = SUMMARY_DIR / "all_10_triple_smote_vs_no_smote_best_per_fs_method.csv"
    df_triple_smote_vs_no_smote_best.to_csv(smote_best_path, index=False)

    print("SMOTE vs no-SMOTE best-per-FS-method table saved:")
    print(smote_best_path)
    display(df_triple_smote_vs_no_smote_best)
else:
    df_triple_smote_vs_no_smote_best = pd.DataFrame()


if "df_full_all_models" in globals() and len(df_full_all_models) > 0:
    df_triple_smote_vs_no_smote_by_model = build_batch_smote_delta_table(
        df_full_all_models,
        key_cols=["Triple_Tag", "Dataset_A", "Dataset_B", "Dataset_C", "Experiment_ID", "FS_METHOD", "Model"]
    )

    smote_by_model_path = SUMMARY_DIR / "all_10_triple_smote_vs_no_smote_by_model.csv"
    df_triple_smote_vs_no_smote_by_model.to_csv(smote_by_model_path, index=False)

    print("SMOTE vs no-SMOTE fixed-model table saved:")
    print(smote_by_model_path)
    display(df_triple_smote_vs_no_smote_by_model)
else:
    df_triple_smote_vs_no_smote_by_model = pd.DataFrame()


display_cols = [
    "Triple_Tag",
    "Dataset_A",
    "Dataset_B",
    "Dataset_C",
    "Rows_After_Merge",
    "N_Features_Total",
    "Label_Counts",
    "Final_Status",
    "FS_METHOD",
    "FS_N_FEATURES",
    "USE_SMOTE",
    "Model",
    "Mean_AUC",
    "Std_AUC",
    "Pooled_AUC",
    "Mean_PR_AUC",
    "Pooled_PR_AUC",
    "Mean_Bal_Acc",
    "Avg_Features",
    "N_Unique_Selected_Features",
    "Run_Name",
    "Result_Error",
]

display(df_all_10_summary[[c for c in display_cols if c in df_all_10_summary.columns]])

In [ ]:
# =============================================================================
# CELL 8 — PUBLICATION-STYLE COMPACT TABLE AND OPTIONAL PLOT
# =============================================================================

publication_cols = [
    "Triple_Tag",
    "Dataset_A",
    "Dataset_B",
    "Dataset_C",
    "Rows_After_Merge",
    "N_Features_Total",
    "Label_Counts",
    "FS_METHOD",
    "FS_N_FEATURES",
    "USE_SMOTE",
    "Model",
    "Mean_AUC",
    "Std_AUC",
    "Pooled_AUC",
    "Mean_PR_AUC",
    "Pooled_PR_AUC",
    "Mean_Bal_Acc",
    "Pooled_Bal_Acc",
    "Mean_F1",
    "Pooled_F1",
    "Avg_Features",
    "N_Unique_Selected_Features",
    "Pooled_AUC_Macro_OvR",
    "Pooled_AUC_Micro_OvR",
    "Pooled_AUC_CI95_Low",
    "Pooled_AUC_CI95_High",
    "Std_PR_AUC",
    "Pooled_PR_AUC_CI95_Low",
    "Pooled_PR_AUC_CI95_High",
    "Mean_Acc",
    "Std_Acc",
    "Pooled_Acc",
    "Pooled_Acc_CI95_Low",
    "Pooled_Acc_CI95_High",
    "Std_Bal_Acc",
    "Pooled_Bal_Acc_CI95_Low",
    "Pooled_Bal_Acc_CI95_High",
    "Mean_Prec",
    "Std_Prec",
    "Pooled_Prec",
    "Pooled_Prec_CI95_Low",
    "Pooled_Prec_CI95_High",
    "Mean_Rec",
    "Std_Rec",
    "Pooled_Rec",
    "Pooled_Rec_CI95_Low",
    "Pooled_Rec_CI95_High",
    "Std_F1",
    "Pooled_F1_CI95_Low",
    "Pooled_F1_CI95_High",
    "Std_Specificity",
    "Pooled_Specificity_CI95_Low",
    "Pooled_Specificity_CI95_High",
    "Mean_Kappa",
    "Std_Kappa",
    "Pooled_Kappa",
    "Pooled_Kappa_CI95_Low",
    "Pooled_Kappa_CI95_High",
    "Pooled_Brier",
    "Pooled_Brier_CI95_Low",
    "Pooled_Brier_CI95_High",
    "CI_Bootstrap_Requested",
    "CI_Bootstrap_Min_Successful",
    "Run_Name",
    "Run_Dir",
    "Final_Status",
    "Result_Error",
]

available_publication_cols = [c for c in publication_cols if c in df_all_10_summary.columns]
df_publication_10_triples = df_all_10_summary[available_publication_cols].copy()

publication_path = SUMMARY_DIR / "publication_all_10_triple_concat_summary.csv"
df_publication_10_triples.to_csv(publication_path, index=False)

print("Publication-style summary saved:")
print(publication_path)

display(df_publication_10_triples)


# Publication-style SMOTE delta table
if "df_triple_smote_vs_no_smote_best" in globals() and len(df_triple_smote_vs_no_smote_best) > 0:
    delta_publication_cols = [
        "Triple_Tag",
        "Dataset_A",
        "Dataset_B",
        "Dataset_C",
        "Experiment_ID",
        "FS_METHOD",
        "Best_Model_No_SMOTE",
        "Best_Model_SMOTE",
        "Mean_AUC_No_SMOTE",
        "Mean_AUC_SMOTE",
        "Delta_Mean_AUC_SMOTE_minus_No_SMOTE",
        "Pooled_AUC_No_SMOTE",
        "Pooled_AUC_SMOTE",
        "Delta_Pooled_AUC_SMOTE_minus_No_SMOTE",
        "Mean_PR_AUC_No_SMOTE",
        "Mean_PR_AUC_SMOTE",
        "Delta_Mean_PR_AUC_SMOTE_minus_No_SMOTE",
        "Mean_Bal_Acc_No_SMOTE",
        "Mean_Bal_Acc_SMOTE",
        "Delta_Mean_Bal_Acc_SMOTE_minus_No_SMOTE",
        "Avg_Features_No_SMOTE",
        "Avg_Features_SMOTE",
        "Run_Name_No_SMOTE",
        "Run_Name_SMOTE",
    ]
    delta_publication_cols = [
        c for c in delta_publication_cols
        if c in df_triple_smote_vs_no_smote_best.columns
    ]
    df_publication_smote_delta = df_triple_smote_vs_no_smote_best[delta_publication_cols].copy()

    delta_publication_path = SUMMARY_DIR / "publication_all_10_triple_smote_vs_no_smote_summary.csv"
    df_publication_smote_delta.to_csv(delta_publication_path, index=False)

    print("Publication-style SMOTE vs no-SMOTE summary saved:")
    print(delta_publication_path)
    display(df_publication_smote_delta)


# Optional simple plot of Mean_AUC by triple combination.
try:
    import matplotlib.pyplot as plt

    plot_df = df_publication_10_triples.copy()
    plot_df["Mean_AUC_numeric"] = pd.to_numeric(plot_df.get("Mean_AUC"), errors="coerce")
    plot_df = plot_df.dropna(subset=["Mean_AUC_numeric"]).sort_values("Mean_AUC_numeric", ascending=True)

    if len(plot_df) > 0:
        fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(plot_df))))
        ax.barh(plot_df["Triple_Tag"], plot_df["Mean_AUC_numeric"])
        ax.set_xlabel("Mean ROC-AUC")
        ax.set_ylabel("Triple")
        ax.set_title("Comparison of all triple-concat datasets")
        ax.set_xlim(0, 1)
        plt.tight_layout()

        plot_path = SUMMARY_DIR / "all_10_triple_mean_auc_barplot.png"
        fig.savefig(plot_path, dpi=300, bbox_inches="tight")
        plt.show()

        print("Plot saved:")
        print(plot_path)

except Exception as e:
    print("Plot skipped:", e)